# Session 23 – Research and Apply Data Analysis with Pandas

**Objective:** Analyze a real dataset, summarize information, and extract meaningful insights using Pandas.

**Dataset:** Iris (150 records, three species). Source: [UCI Machine Learning Repository – Iris](https://archive.ics.uci.edu/dataset/53/iris). The included CSV is the standard Iris dataset distributed in scikit-learn, with readable column names and species labels.

## Part 1: Research

### 1. How do filtering and sorting work in Pandas?
**Filtering** selects only rows that satisfy a condition. A condition such as `df['age'] > 18` creates a Boolean mask, and using `df[mask]` returns matching rows. Multiple conditions can be combined with `&` (AND) or `|` (OR); each condition should be wrapped in parentheses. **Sorting** rearranges rows by one or more columns using `sort_values()`. The `ascending` argument controls the direction, and `sort_index()` sorts by row or column labels.

### 2. What is the purpose of `groupby()`?
`groupby()` divides rows into groups that share the same value (or combination of values) in selected columns. We can then calculate a summary for each group. This is commonly described as split–apply–combine: split the data, apply a calculation, and combine the results.

### 3. What are aggregation functions such as `sum()`, `mean()`, `count()`, and `max()`?
Aggregation functions reduce multiple values to summary values. `sum()` adds values, `mean()` calculates their arithmetic average, `count()` counts non-missing values, and `max()` returns the largest value. They can be used on a whole column or after `groupby()` to compare groups.

### 4. What is the difference between `merge()` and `concat()`?
`merge()` combines DataFrames by matching key columns or indexes, similar to a database join. It supports join types such as `inner`, `left`, `right`, and `outer`. `concat()` stacks DataFrames along rows (`axis=0`, the default) or columns (`axis=1`) without matching records by key. Use `merge()` for related tables with a common key and `concat()` when appending or aligning pieces of data.

### 5. What is a Pandas `pivot_table()`, and when is it useful?
`pivot_table()` reshapes data into a spreadsheet-style summary. It groups by one or more index fields and calculates an aggregation (mean by default) for selected values. It is useful for comparing measures across categories and can summarize data using multiple row/column categories.

## Part 2: Practical Application

### 1. Load the dataset and inspect its structure

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path.cwd()
df = pd.read_csv(DATA_DIR / 'iris_dataset.csv')
print('Shape:', df.shape)
display(df.head())
df.info()
display(df.describe())

### 2. Filter records using at least two conditions

In [ ]:
# Keep flowers whose petal length is above 4.5 cm AND sepal length is above 6 cm.
filtered_df = df[(df['petal_length_cm'] > 4.5) & (df['sepal_length_cm'] > 6.0)]
print('Matching records:', len(filtered_df))
display(filtered_df.head())

### 3. Sort records by selected columns

In [ ]:
sorted_df = df.sort_values(
    by=['species', 'petal_length_cm'],
    ascending=[True, False]
)
display(sorted_df.head(10))

### 4. Use `groupby()` with aggregation functions

In [ ]:
group_summary = df.groupby('species').agg(
    sample_count=('sample_id', 'count'),
    mean_petal_length=('petal_length_cm', 'mean'),
    max_sepal_length=('sepal_length_cm', 'max'),
    mean_sepal_width=('sepal_width_cm', 'mean')
).round(2)
display(group_summary)

### 5. Combine two related DataFrames using `merge()`

In [ ]:
species_info = pd.read_csv(DATA_DIR / 'species_info.csv')
merged_df = df.merge(
    species_info,
    on=['species_id', 'species'],
    how='left',
    validate='many_to_one'
)
print('Merged shape:', merged_df.shape)
display(merged_df.head())

### 6. Create a `pivot_table()`

In [ ]:
pivot = pd.pivot_table(
    df,
    index='species',
    values=['sepal_length_cm', 'petal_length_cm'],
    aggfunc='mean'
).round(2)
display(pivot)

### 7. At least three findings from the analysis

The following values are calculated from the dataset rather than guessed.

In [ ]:
petal_means = df.groupby('species')['petal_length_cm'].mean()
species_counts = df.groupby('species')['sample_id'].count()

print(f"1. Virginica has the largest average petal length: {petal_means['virginica']:.2f} cm.")
print(f"2. Setosa has the smallest average petal length: {petal_means['setosa']:.2f} cm.")
print(f"3. The dataset is balanced: each species has {int(species_counts.iloc[0])} samples.")

## Conclusion

Pandas filtering and sorting help select and organize records. `groupby()` and aggregation provide category-level summaries, `merge()` adds information from a related lookup table, and `pivot_table()` presents compact comparisons. In this dataset, petal length clearly distinguishes the three species: setosa has the smallest mean and virginica the largest.